# Notebook 02: Coleta e Pré-Visualização (Staging em Memória)

**Fonte dos dados:** `scikit-learn` – *Breast Cancer Wisconsin (Diagnostic)*  
- Dataset acadêmico amplamente utilizado para análises de classificação.
- Contém atributos numéricos extraídos de imagens de biópsias de mama.
- A variável alvo (**diagnosis**) indica se o tumor é **benigno (0)** ou **maligno (1)**.

**Objetivo deste notebook:**  
1. Coletar o dataset a partir de uma fonte confiável (scikit-learn).  
2. Realizar inspeção inicial (schema + amostra) para validar estrutura.  
3. Registrar um ponto importante: os nomes originais possuem espaços, o que exige padronização antes de persistir em Delta tables (será feito no Bronze).

In [0]:

from sklearn.datasets import load_breast_cancer
import pandas as pd

# Carregamento do dataset a partir do scikit-learn
data = load_breast_cancer()

# Transformação inicial para DataFrame pandas (visualização mais amigável)
df_pd = pd.DataFrame(data.data, columns=data.feature_names)
df_pd["diagnosis"] = data.target

df_pd.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


### Entendendo a coluna `diagnosis`
No dataset do scikit-learn, o campo `diagnosis` é numérico:
- `0` = benigno  
- `1` = maligno  

Na camada **Silver**, criaremos uma coluna descritiva (`diagnosis_desc`) para facilitar consultas e leitura.

In [0]:
# Conversão para Spark DataFrame (padrão do Databricks/Spark)
df_spark = spark.createDataFrame(df_pd)

# Inspeção do schema para evidenciar tipos e nulabilidade
df_spark.printSchema()

#Visualização
display(df_spark.limit(10))

root
 |-- mean radius: double (nullable = true)
 |-- mean texture: double (nullable = true)
 |-- mean perimeter: double (nullable = true)
 |-- mean area: double (nullable = true)
 |-- mean smoothness: double (nullable = true)
 |-- mean compactness: double (nullable = true)
 |-- mean concavity: double (nullable = true)
 |-- mean concave points: double (nullable = true)
 |-- mean symmetry: double (nullable = true)
 |-- mean fractal dimension: double (nullable = true)
 |-- radius error: double (nullable = true)
 |-- texture error: double (nullable = true)
 |-- perimeter error: double (nullable = true)
 |-- area error: double (nullable = true)
 |-- smoothness error: double (nullable = true)
 |-- compactness error: double (nullable = true)
 |-- concavity error: double (nullable = true)
 |-- concave points error: double (nullable = true)
 |-- symmetry error: double (nullable = true)
 |-- fractal dimension error: double (nullable = true)
 |-- worst radius: double (nullable = true)
 |-- worst 

mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,0
20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.0186,0.0134,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.186,0.275,0.08902,0
19.69,21.25,130.0,1203.0,0.1096,0.1599,0.1974,0.1279,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.00615,0.04006,0.03832,0.02058,0.0225,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.243,0.3613,0.08758,0
11.42,20.38,77.58,386.1,0.1425,0.2839,0.2414,0.1052,0.2597,0.09744,0.4956,1.156,3.445,27.23,0.00911,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.5,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.173,0
20.29,14.34,135.1,1297.0,0.1003,0.1328,0.198,0.1043,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.01149,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.2,1575.0,0.1374,0.205,0.4,0.1625,0.2364,0.07678,0
12.45,15.7,82.57,477.1,0.1278,0.17,0.1578,0.08089,0.2087,0.07613,0.3345,0.8902,2.217,27.19,0.00751,0.03345,0.03672,0.01137,0.02165,0.005082,15.47,23.75,103.4,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.1244,0
18.25,19.98,119.6,1040.0,0.09463,0.109,0.1127,0.074,0.1794,0.05742,0.4467,0.7732,3.18,53.91,0.004314,0.01382,0.02254,0.01039,0.01369,0.002179,22.88,27.66,153.2,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368,0
13.71,20.83,90.2,577.9,0.1189,0.1645,0.09366,0.05985,0.2196,0.07451,0.5835,1.377,3.856,50.96,0.008805,0.03029,0.02488,0.01448,0.01486,0.005412,17.06,28.14,110.6,897.0,0.1654,0.3682,0.2678,0.1556,0.3196,0.1151,0
13.0,21.82,87.5,519.8,0.1273,0.1932,0.1859,0.09353,0.235,0.07389,0.3063,1.002,2.406,24.32,0.005731,0.03502,0.03553,0.01226,0.02143,0.003749,15.49,30.73,106.2,739.3,0.1703,0.5401,0.539,0.206,0.4378,0.1072,0
12.46,24.04,83.97,475.9,0.1186,0.2396,0.2273,0.08543,0.203,0.08243,0.2976,1.599,2.039,23.94,0.007149,0.07217,0.07743,0.01432,0.01789,0.01008,15.09,40.68,97.65,711.4,0.1853,1.058,1.105,0.221,0.4366,0.2075,0


In [0]:
# Tamanho do dataset (evidência rápida)
print("Linhas:", df_spark.count())
print("Colunas:", len(df_spark.columns))

# Estatísticas descritivas iniciais (evidência de inspeção dos dados)
display(df_spark.describe())

Linhas: 569
Colunas: 31


summary,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
count,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569
mean,14.127291739894552,19.289648506151142,91.96903339191563,654.8891036906855,0.09636028119507907,0.10434098418277678,0.0887993158172232,0.04891914586994729,0.18116186291739897,0.06279760984182776,0.4051720562390158,1.2168534270650264,2.8660592267135323,40.337079086116,0.007040978910369067,0.025478138840070295,0.03189371634446397,0.011796137082601054,0.02054229876977153,0.0037949038664323374,16.269189806678384,25.677223198594024,107.26121265377856,880.5831282952548,0.1323685940246046,0.2542650439367311,0.27218848330404216,0.11460622319859401,0.2900755711775044,0.0839458172231986,0.6274165202108963
stddev,3.5240488262120775,4.301035768166949,24.2989810387549,351.9141291816529,0.014064128137673614,0.0528127579325122,0.07971980870789348,0.038802844859153605,0.027414281336035715,0.007060362795084458,0.2773127329861039,0.5516483926172023,2.0218545540421076,45.4910055161318,0.003002517943839066,0.017908179325677388,0.03018606032298841,0.006170285174046869,0.008266371528798397,0.002646070967089195,4.833241580469323,6.14625762303832,33.602542269036356,569.356992669949,0.022832429404835458,0.15733648891374197,0.20862428060813232,0.06573234119594207,0.061867467537518685,0.018061267348893986,0.4839179564031687
min,6.981,9.71,43.79,143.5,0.05263,0.01938,0.0,0.0,0.106,0.04996,0.1115,0.3602,0.757,6.802,0.001713,0.002252,0.0,0.0,0.007882,8.948E-4,7.93,12.02,50.41,185.2,0.07117,0.02729,0.0,0.0,0.1565,0.05504,0
max,28.11,39.28,188.5,2501.0,0.1634,0.3454,0.4268,0.2012,0.304,0.09744,2.873,4.885,21.98,542.2,0.03113,0.1354,0.396,0.05279,0.07895,0.02984,36.04,49.54,251.2,4254.0,0.2226,1.058,1.252,0.291,0.6638,0.2075,1


### Observação técnica importante (para o pipeline)
Os nomes originais das colunas contêm **espaços** (ex.: `mean radius`).  
Ao persistir como tabela Delta no Databricks, isso pode gerar erro de compatibilidade.

✅ Portanto, a padronização dos nomes de colunas será feita na camada **Bronze**, antes da persistência.